<a href="https://colab.research.google.com/github/PrathamTumminakatti/ML1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathamTumminakatti/ML1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the search performance of a single content item for a specific client on a specific report date. The analysis will use the `fact_content_daily_performance` table (or the `_sample` table for iteration), where the grain is `report_date × client × content`.

## Time Window

For this assignment, a mid-panel month (2026-03) will be used for verification queries and feature construction. The final month (2026-06) is treated as a sealed test period and will not be used for developing labels or features.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# Fields: Feature / Label / Context / Excluded

## Features
These fields are available before the prediction period and can be used for analysis or model development.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- scroll_events

## Label
The target variable to be predicted. It is **not** included in this daily performance table and should be created separately from future performance when building a machine learning model.

## Context
These fields identify records or provide metadata but are **not** used as model features.

- report_date
- client_hash_id
- content_hash_id
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

## Excluded
These fields are intentionally excluded from model training.

- client_hash_id – Identifier used only for grouping, joining, and train/test splitting.
- content_hash_id – Identifier used only for grouping and joining.
- report_date – Used for defining time windows, not as a predictive feature.
- Any future-derived or label-derived fields (if created later) – Excluded to prevent data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
import duckdb

con = duckdb.connect()

In [21]:
sample_files = [f for f in files if "fact_content_daily_performance_sample" in f]
print(sample_files)

['fact_content_daily_performance_sample.parquet']


In [22]:
from huggingface_hub import hf_hub_download

sample_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print(sample_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet


In [23]:
import duckdb

con = duckdb.connect()

con.sql(f"""
CREATE OR REPLACE VIEW performance AS
SELECT *
FROM read_parquet('{sample_path}');
""")

print("✅ View created successfully!")

✅ View created successfully!


In [24]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM performance;
""").df()


,total_rows,first_date,last_date
0,11694072,2026-06-01,2026-06-30


In [25]:
con.sql("""
DESCRIBE performance;
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [26]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM performance
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-13,client_e00b29e582949543,content_e3491394a9f3e2b3,2
1,2026-06-13,client_e00b29e582949543,content_83b20e2cd4d5437f,2
2,2026-06-13,client_e00b29e582949543,content_04307a138472fc03,2
3,2026-06-13,client_e00b29e582949543,content_638ac5ad8023d177,2
4,2026-06-13,client_e00b29e582949543,content_ff881e76e85001f4,2


In [27]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM performance;
""").df()

,total_rows,first_date,last_date
0,11694072,2026-06-01,2026-06-30


In [28]:
con.sql("""
SELECT
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS missing_gsc_avg_position,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS missing_gsc_clicks,
    AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS missing_ga4_sessions
FROM performance;
""").df()

,missing_gsc_avg_position,missing_gsc_clicks,missing_ga4_sessions
0,0.668302,0.0,0.205012


In [29]:
con.sql("""
SELECT
    month,
    COUNT(*) AS rows_per_month
FROM performance
GROUP BY month
ORDER BY month;
""").df()

,month,rows_per_month
0,2026-06,11694072


### Verification Summary

The executed queries verify that:

- The dataset grain is one row per `report_date × client_hash_id × content_hash_id`.
- The available reporting period matches the documented warehouse release.
- Key fields such as `gsc_avg_position`, `gsc_clicks`, and `ga4_sessions` were checked for missing values.
- The monthly distribution confirms the available reporting windows for analysis.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

# Data Limits

This dataset has several important limitations that must be considered during analysis:

- **Uneven history across clients:** Different clients have different amounts of historical data, so comparisons across all clients may not always be fair.
- **Incomplete analytics coverage:** Some early records contain Google Search Console (GSC) data but do not have Google Analytics 4 (GA4) data. The `ga4_data_available` flag should be checked before using GA4 metrics.
- **Missing values are not random:** Some fields may be unavailable for certain records, so missing values should be handled carefully instead of blindly replacing them with zeros.
- **Time-window overlap:** Features and labels must be created using separate time windows to prevent data leakage.
- **Decision-support only:** This dataset can identify patterns in historical search performance, but it cannot prove that one metric directly causes another or guarantee future performance.

In [30]:
!pip -q install duckdb huggingface_hub pandas pyarrow

In [31]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "Token not found.")

Token loaded successfully!


In [32]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [33]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print(f"Found {len(files)} files")
print(files[:10])

Found 24 files
['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet']


In [34]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM performance;
""").df()

,total_rows,first_date,last_date
0,11694072,2026-06-01,2026-06-30


In [35]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM performance
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-13,client_e00b29e582949543,content_e3491394a9f3e2b3,2
1,2026-06-13,client_e00b29e582949543,content_83b20e2cd4d5437f,2
2,2026-06-13,client_e00b29e582949543,content_04307a138472fc03,2
3,2026-06-13,client_e00b29e582949543,content_638ac5ad8023d177,2
4,2026-06-13,client_e00b29e582949543,content_ff881e76e85001f4,2


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.